# Reconstruct pool state block-by-block

Turn the irregular swap events into one aligned, gap-free per-pool series
(price + active liquidity), trim to the study window, and save one CSV per pool
under `S.processed_dir`. Parameters come from `arblib.config.STUDY`.

In [ ]:
%load_ext autoreload
%autoreload 2

from arblib import data_io, preprocessing as pp
from arblib import formulas as f
from arblib import modeling, regime
from arblib.config import STUDY as S, LIQUIDITY_FILES

REGIME = "high"                      # same window as extract: "low" | "mid" | "high"
STUDY_START = regime.study_window(S, REGIME)["study_start"]
print("processing regime:", REGIME, "| study starts:", STUDY_START)

## 0. Load the raw swap extracts

In [7]:
dfs = data_io.load_pool_csvs(S.swaps_dir)

Loaded: df_uniswap_swap.csv
       amount0               amount1         dex  evt_block_number  \
0   -999266400    334197952309227723  uniswap_v3          24133363   
1     33925490    -11345521500000000  uniswap_v4          24133364   
2  13936469654  -4658615254546375464  uniswap_v3          24133365   
3   -107703773     36025562985474874  uniswap_v3          24133365   
4   -154244910     51582178535019641  uniswap_v4          24133365   

                evt_block_time  evt_index  \
0  2025-12-31 15:00:11.000 UTC         17   
1  2025-12-31 15:00:23.000 UTC          3   
2  2025-12-31 15:00:35.000 UTC        353   
3  2025-12-31 15:00:35.000 UTC         40   
4  2025-12-31 15:00:35.000 UTC        719   

                                         evt_tx_hash  fee  \
0  0x045ccf214364711f41a1e4ebdc460703fb1d3f6ccfae...  100   
1  0x7e40441f67a6ffeb29115fdf3028481faf2a2b9772bf...    0   
2  0xa8e2974554c1b06a7ab79cf43f0764f29ea521b0e1e6...  500   
3  0xa0671b25449576f08044498e74206b1

## 1. Keep the end-of-block price per pool & block

In [8]:
dfs = pp.count_swaps(dfs)
filtered_dfs = pp.clean_all(dfs)

Counted swaps df_uniswap: 1066 rows
Counted swaps df_pancake: 240 rows
Processed df_uniswap: 1066 rows -> 605 rows
Processed df_pancake: 240 rows -> 180 rows


## 2. Split each DEX into one series per pool

In [9]:
pool_dfs = pp.split_by_pool(filtered_dfs)

Created uniswap_1: 75 rows
Created uniswap_2: 128 rows
Created uniswap_3: 24 rows
Created uniswap_4: 3 rows
Created uniswap_5: 230 rows
Created uniswap_6: 145 rows
Created pancake_1: 137 rows
Created pancake_2: 43 rows

Total: 8 dataframes
['uniswap_1', 'uniswap_2', 'uniswap_3', 'uniswap_4', 'uniswap_5', 'uniswap_6', 'pancake_1', 'pancake_2']


## 3. Drop pools that trade too rarely to reconstruct

In [10]:
pool_dfs, dropped_pools = pp.filter_pools_by_swap_gap(pool_dfs, S.max_gap_blocks)

k = 6000 blocks
Kept 8 pools

Kept pools:
  uniswap_1: 75 swaps, max consecutive gap = 16 blocks
  uniswap_2: 128 swaps, max consecutive gap = 11 blocks
  uniswap_3: 24 swaps, max consecutive gap = 49 blocks
  uniswap_4: 3 swaps, max consecutive gap = 107 blocks
  uniswap_5: 230 swaps, max consecutive gap = 5 blocks
  uniswap_6: 145 swaps, max consecutive gap = 9 blocks
  pancake_1: 137 swaps, max consecutive gap = 9 blocks
  pancake_2: 43 swaps, max consecutive gap = 24 blocks


## 3b. Drop dynamic-fee pools

The execution-price math needs a single fixed fee per pool, so pools whose `fee`
varies (e.g. Uniswap v4 dynamic-fee hooks) are excluded before saving.

In [11]:
pool_dfs, dropped_fee_pools = pp.filter_pools_by_constant_fee(pool_dfs)

Kept 7 constant-fee pools
Dropped 1 dynamic-fee pool(s):
  uniswap_6: fees = [0, 238]


## 4. Reconstruct a dense, forward-filled series per pool

In [12]:
reconstructed_pools, global_min, global_max, block_time_map = pp.reconstruct_pool_timeseries(pool_dfs)

Global block range: 24133363 to 24133662
Total blocks: 300

uniswap_1: 75 trades -> 295 blocks (98.3%)
uniswap_2: 128 trades -> 298 blocks (99.3%)
uniswap_3: 24 trades -> 297 blocks (99.0%)
uniswap_4: 3 trades -> 235 blocks (78.3%)
uniswap_5: 230 trades -> 300 blocks (100.0%)
pancake_1: 137 trades -> 298 blocks (99.3%)
pancake_2: 43 trades -> 290 blocks (96.7%)

Created 7 reconstructed time series


## 4b. Reconstruct active liquidity per block

Rebuild each pool's `liquidity` into the running active-liquidity state using the
mint/burn events (in-range deltas applied between swaps, held constant otherwise).

In [13]:
liq_dfs = data_io.load_pool_csvs(S.liquidity_dir, files=LIQUIDITY_FILES)
reconstructed_pools = pp.reconstruct_liquidity_states(reconstructed_pools, liq_dfs)

Loaded: df_uniswap_liq.csv
          dex  evt_block_number               evt_block_time  evt_index  \
0  uniswap_v3          24133387  2025-12-31 15:04:59.000 UTC        557   
1  uniswap_v3          24133406  2025-12-31 15:08:47.000 UTC        179   
2  uniswap_v3          24133460  2025-12-31 15:19:35.000 UTC        318   
3  uniswap_v3          24133591  2025-12-31 15:45:47.000 UTC        141   
4  uniswap_v3          24133626  2025-12-31 15:52:47.000 UTC        198   

        liquidity_delta                                        pool  \
0        -1291771469796  0xe0554a476a092703abdb3ef35c80e0d76d32939f   
1      5244061520071531  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640   
2  28535564165621841753  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640   
3       -49393047801584  0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640   
4      -509787476461393  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   

   tick_lower  tick_upper  
0      194973      197604  
1      196250      196730  
2      1963

## 5a. MEV friction proxies

Two decay-weighted MEV series per pool: `mev_intensity` (recent top priority tip,
`gas_price_max - base_fee`) and `contest_frequency` (recent rate of same-block races,
`nb_swaps >= 2`), each decayed over past swap blocks with horizon `S.mev_horizon_blocks`.
No forward-fill — a quiet block inherits no stale competition value (gets `NaN`).

In [14]:
chain_gas = data_io.load_chain_gas(S.gas_path)
reconstructed_pools = f.add_mev_intensity(reconstructed_pools, chain_gas, S.mev_horizon_blocks)
reconstructed_pools = f.add_contest_freq(reconstructed_pools, S.mev_horizon_blocks)

uniswap_1: mev_intensity max 9.275e+08 wei
uniswap_2: mev_intensity max 1.445e+10 wei
uniswap_3: mev_intensity max 8.424e+09 wei
uniswap_4: mev_intensity max 5.026e+08 wei
uniswap_5: mev_intensity max 1.879e+10 wei
pancake_1: mev_intensity max 1.434e+10 wei
pancake_2: mev_intensity max 1.427e+10 wei
uniswap_1: nb_swaps_ewma max 0.92
uniswap_2: nb_swaps_ewma max 1.59
uniswap_3: nb_swaps_ewma max 0.35
uniswap_4: nb_swaps_ewma max 0.09
uniswap_5: nb_swaps_ewma max 3.25
pancake_1: nb_swaps_ewma max 1.13
pancake_2: nb_swaps_ewma max 0.46


## 5. Trim to the study window

In [15]:
filtered_pools = pp.filter_by_start_time(reconstructed_pools, STUDY_START)

uniswap_1: 300 -> 225 rows
uniswap_2: 300 -> 225 rows
uniswap_3: 300 -> 225 rows
uniswap_4: 300 -> 225 rows
uniswap_5: 300 -> 225 rows
pancake_1: 300 -> 225 rows
pancake_2: 300 -> 225 rows

Filtered all pools by time >= 2025-12-31 15:15:00


## 6. Save one CSV per pool

In [16]:
data_io.save_processed_pools(filtered_pools, S.processed_dir)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_2.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_3.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_4.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/uniswap_5.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/pancake_1.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/WETH_USDC/processed/pancake_2.csv
Done.


In [18]:
pancake_1_metrics = filtered_pools['uniswap_2'][[
    'nb_swaps',
    'gas_price_max',
    'mev_intensity',
    'nb_swaps_ewma'
]]
pancake_1_metrics.head(25)

,nb_swaps,gas_price_max,mev_intensity,nb_swaps_ewma
0,1,1.524613e+08,9.639456e+09,1.107606
1,0,NaN,9.017778e+09,1.036173
2,0,NaN,8.436194e+09,0.969348
3,0,NaN,7.892119e+09,0.906831
4,1,1.558764e+08,7.383197e+09,0.912840
5,0,NaN,6.907032e+09,0.853968
6,0,NaN,6.461577e+09,0.798893
7,1,1.645411e+08,6.044850e+09,0.811863
8,1,1.577591e+08,5.655000e+09,0.823997
9,0,NaN,5.290292e+09,0.770855


## 7. Save the common (pool-independent) modeling covariates

Persist the covariates every pool pair shares to `S.common_covariates_dir`
(`modeling/covariates/common_covariates/`):

- **`CEX_volatility.parquet`** — `[time, ewma_vol]`: RiskMetrics EWMA volatility of the
  `token0/token1` exchange rate `R = P_X/USD / P_Y/USD` (log returns differenced over time,
  `var_t = λ·var_{t-1} + (1-λ)·r_t²`, `λ = exp(-1/S.vol_horizon_min)`). Minute grid — joined to
  blocks by a backward merge on `time` downstream.
- **`chain_covariates.parquet`** — `[block_number, time, log_base_fee_per_gas, gas_util,
  log1p_tip_p50, log1p_tip_p90]`: base fee in logs, block fullness `gas_used/gas_limit`, and the
  per-block priority-tip p50/p90 as `log(1+tip)`. Joined by exact `block_number`.

Also creates (empty) `pool_pair_dependant_covariates/` for the per-pool-pair features built
later. Both files span the full extract window (warm-up included); downstream joins select the
study blocks.

In [ ]:
x_usd = data_io.load_price_series(S.x_price_path)
y_usd = data_io.load_price_series(S.y_price_path)

modeling.save_common_covariates(x_usd, y_usd, chain_gas, S.common_covariates_dir, S.vol_horizon_min)
S.pair_covariates_dir.mkdir(parents=True, exist_ok=True)

mev competition : max tip/block ->esssayer median tip ? ou autre quantiles 

-evenement multiple fenetre en fonctio de vol : petite/moy/big